In [1]:
import numpy as np
from typing import Dict
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import torch.nn.functional as F
import math

In [2]:
dropout = 0.1
lr=1e-3
weight_decay= 1e-4
batch_size = 256
epochs  = 20 
seed  = 24
device = "cuda:1" if torch.cuda.is_available() else "cpu"

In [3]:
import pandas as pd
data=pd.read_csv("../example_data/all_means.csv")

In [4]:
data

,gene,CK_mean,HS_mean,loop_ckmean,loop_hsmean,K4_ckmean,K4_hsmean,K27_ckmean,K27_hsmean,atac_ckmean,atac_hsmean,com_ckmean,com_hsmean
0,Zm00001d020750,0.000000,0.000000,-0.478295,-0.580962,-0.184387,0.396443,-0.207874,-0.119462,-0.069284,-0.056039,0.463896,0.186753
1,Zm00001d007947,4.539262,5.095064,-0.148191,-0.266176,-0.384754,-0.340036,-0.242612,-0.440865,-0.186460,-0.170282,0.177135,0.212616
2,Zm00001d038248,5.861395,5.988716,-0.432447,-0.349502,-0.245905,-0.151559,-0.188722,-0.130280,0.021330,-0.049925,0.820748,0.344796
3,Zm00001d038249,0.000000,0.096025,-0.361383,-0.199053,1.825752,1.637270,0.064147,-0.159594,0.024450,0.020907,0.820748,0.344796
4,Zm00001d013244,0.019426,1.698778,0.601419,0.884181,-0.589677,-0.770743,-0.230029,-0.401647,0.170977,0.042714,0.591313,0.166271
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21407,Zm00001d026149,1.655748,1.909210,-1.055977,-0.942040,0.000000,0.000000,0.000000,0.000000,-0.017010,-0.028464,0.380464,-0.456974
21408,Zm00001d044514,4.709129,5.354951,-0.382015,-0.178221,-0.902593,-0.826512,-0.232565,-0.261242,-0.161105,-0.132449,0.830872,0.590943
21409,Zm00001d026144,4.495102,4.637174,-1.000959,-0.747614,0.000000,0.000000,0.000000,0.000000,-0.115210,-0.129248,0.190291,-0.009251
21410,Zm00001d026147,4.188590,2.338468,-0.492049,-0.303210,0.000000,0.000000,0.000000,0.000000,-0.046262,-0.076978,0.683245,-0.166170


create dataloader

In [5]:
def split_indices(n: int, seed: int = seed) -> Dict[str, np.ndarray]:
    """
    train_idx(64%) / loop_idx(16%) / test_idx(20%)
    """
    rng = np.random.default_rng(seed)
    all_idx = np.arange(n)
    rng.shuffle(all_idx)
    n_train_zone = int(0.8 * n)
    train_zone = all_idx[:n_train_zone]    # 80%
    test_idx   = all_idx[n_train_zone:]    # 20%
    n_train = int(0.8 * train_zone.size)   # 80% of 80% = 64%
    train_idx = train_zone[:n_train]
    loop_idx  = train_zone[n_train:]       # 20% of 80% = 16%
    return dict(train=train_idx, loop=loop_idx, test=test_idx)

In [6]:
splits = split_indices(len(data),seed=seed)
tr_idx, loop_idx, te_idx = splits["train"], splits["loop"], splits["test"]

In [7]:
class multiDataset(Dataset):
    def __init__(self,mat,feature,label):
        self.gene=mat["gene"].values
        self.mat=torch.as_tensor(mat[feature].values,dtype=torch.float32)
        self.label=torch.as_tensor(mat[label].values,dtype=torch.float32)
    def __len__(self):
        return len(self.label)

    def __getitem__(self,i):
        return self.mat[i],self.label[i],self.gene[i]


In [8]:
train_data=data.loc[tr_idx]
valid_data=data.loc[loop_idx]
test_data=data.loc[te_idx]

In [9]:
feature_cols = data.columns.tolist()[1::2]
label_6=data.columns.tolist()[2::2]
ds_train=multiDataset(train_data,feature_cols,label_6) 
dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)

ds_valid=multiDataset(valid_data,feature_cols,label_6) 
dl_valid = DataLoader(ds_valid, batch_size=batch_size, shuffle=True)

ds_test=multiDataset(test_data,feature_cols,label_6) 
dl_test = DataLoader(ds_test, batch_size=batch_size, shuffle=True)

In [10]:
for ma,la,gene in dl_train:
    print(ma.shape)
    print(la.shape)
    print(len(gene))
    print(la.size(0))
    num_elements = la.numel()
    print(num_elements)
    print(gene[1])
    break

torch.Size([256, 6])
torch.Size([256, 6])
256
256
1536
Zm00001d017657


main model

In [11]:
class net(nn.Module):
    def __init__(self,dropout):
        super().__init__()
        self.stru=nn.Sequential(
            nn.Linear(6,36),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(36,12),
            nn.ReLU(),
            nn.Linear(12,6)
        )
    def forward(self,mat):
        value=self.stru(mat)
        return value

train

In [12]:
def calc_pearson(la_true, la_pred):
    la_true = np.asarray(la_true).reshape(-1)
    la_pred = np.asarray(la_pred).reshape(-1)

    if len(la_true) < 2:
        return np.nan
    if np.std(la_true) == 0 or np.std(la_pred) == 0:
        return np.nan

    return np.corrcoef(la_true,la_pred)[0, 1]

In [13]:
def run_one_epoch(model,loader,optimizer,device,train=True):
    model.train(mode=train)
    total_loss=0.0
    total_sq_error=0.0
    total_abs_error=0.0
    total = 0
    all_preds=[]
    all_labels=[]
    for data,label,_ in loader:
        data=data.to(device)
        label=label.to(device).view(-1, 6).float()
        pred=model(data)
        loss = F.smooth_l1_loss(pred,label)
        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        diff=pred-label

        num_elements = label.numel()
        total_loss += loss.item() * num_elements
        total_sq_error += torch.sum(diff ** 2).item()
        total_abs_error += torch.sum(torch.abs(diff)).item()
        total += num_elements

        all_preds.append(pred.detach().cpu().numpy())
        all_labels.append(label.detach().cpu().numpy())
    avg_loss = total_loss / max(total, 1)
    rmse = math.sqrt(total_sq_error / max(total, 1))
    mae = total_abs_error / max(total, 1)

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    pearson = []
    for i in range(all_preds.shape[1]):
        pearson.append(calc_pearson(all_labels[:, i], all_preds[:, i]))
    return avg_loss, rmse, mae, pearson

In [14]:
def train_model(model,dl_train,dl_valid,device,to):
    torch.manual_seed(seed); np.random.seed(seed)
    device=torch.device(device)
    optim=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=weight_decay)
    
    best_path = to
    best_rmse = 100
    best_loss=100
    min_diff=1e-3
    no_improve=0
    num_no_improve = 5
    ep_idx=0

    f_train_loss=0
    f_train_rmse=0
    f_train_mae=0
    f_train_pearson=0
    f_val_loss=0
    f_val_rmse=0
    f_val_mae=0
    f_val_pearson=0

    for ep in range(1,epochs+1):
        train_loss, train_rmse, train_mae, train_pearson = run_one_epoch(model, dl_train, optim, device, train=True)
        val_loss, val_rmse, val_mae, val_pearson = run_one_epoch(model, dl_valid,optimizer=None, device=device, train=False)
        pearson_mean1 = float(np.mean(train_pearson))
        pearson_mean2 = float(np.mean(val_pearson))
        print(f"[Epoch_{ep:02d}] "
            f"train_loss {train_loss:.4f} rmse {train_rmse:.4f} "
            f"train_mae {train_mae:.4f} train_pearson_mean {pearson_mean1:.4f} "
            f"valid_loss {val_loss:.4f} rmse {val_rmse:.4f} "
            f"valid_mae {val_mae:.4f} val_pearson_mean {pearson_mean2:.4f} ")
    
        if val_rmse  < best_rmse:
            best_rmse=val_rmse
            torch.save(model.state_dict(), best_path)
            ep_idx=ep
            f_train_loss=train_loss
            f_train_rmse=train_rmse
            f_train_mae=train_mae
            f_train_pearson=train_pearson
            f_val_loss=val_loss
            f_val_rmse=val_rmse
            f_val_mae=val_mae
            f_val_pearson=val_pearson

        if best_loss - val_loss > min_diff:
            best_loss = val_loss
            no_improve = 0
        else:
            no_improve +=1
            if no_improve > num_no_improve:
                print(f"Early stopping at epoch {ep}")
                break
    pearson_mean3 = float(np.mean(f_train_pearson))
    pearson_mean4 = float(np.mean(f_val_pearson))
    print(f"final saved model at {ep_idx}")
    print(f"[Epoch_{ep_idx:02d}] "
            f"train_loss {f_train_loss:.4f} rmse {f_train_rmse:.4f} "
            f"train_mae {f_train_mae:.4f} train_pearson_mean {pearson_mean3:.4f} "
            f"valid_loss {f_val_loss:.4f} rmse {f_val_rmse:.4f} "
            f"valid_mae {f_val_mae:.4f} val_pearson_mean {pearson_mean4:.4f} ")
    return f_train_loss,f_train_rmse,f_train_mae,f_train_pearson,f_val_loss,f_val_rmse,f_val_mae,f_val_pearson

In [15]:
model=net(dropout=dropout).to(device)

In [16]:
f_train_loss,f_train_rmse,f_train_mae,f_train_pearson,f_val_loss,f_val_rmse,f_val_mae,f_val_pearson = \
train_model(model,dl_train,dl_valid,device,"best_model_CKpredHS_rep1.pt")

[Epoch_01] train_loss 0.5767 rmse 1.5425 train_mae 0.8703 train_pearson_mean 0.4308 valid_loss 0.4974 rmse 1.3749 valid_mae 0.7844 val_pearson_mean 0.6527 
[Epoch_02] train_loss 0.3632 rmse 1.1018 train_mae 0.6457 train_pearson_mean 0.6095 valid_loss 0.1955 rmse 0.7069 valid_mae 0.4592 val_pearson_mean 0.7177 
[Epoch_03] train_loss 0.1694 rmse 0.6772 train_mae 0.4216 train_pearson_mean 0.7134 valid_loss 0.1249 rmse 0.5737 valid_mae 0.3499 val_pearson_mean 0.7631 
[Epoch_04] train_loss 0.1331 rmse 0.6149 train_mae 0.3559 train_pearson_mean 0.7552 valid_loss 0.1063 rmse 0.5393 valid_mae 0.3039 val_pearson_mean 0.8093 
[Epoch_05] train_loss 0.1221 rmse 0.5883 train_mae 0.3315 train_pearson_mean 0.7832 valid_loss 0.1010 rmse 0.5166 valid_mae 0.2937 val_pearson_mean 0.8273 
[Epoch_06] train_loss 0.1171 rmse 0.5627 train_mae 0.3233 train_pearson_mean 0.7985 valid_loss 0.0964 rmse 0.4894 valid_mae 0.2856 val_pearson_mean 0.8379 
[Epoch_07] train_loss 0.1106 rmse 0.5290 train_mae 0.3144 train_

validation on test data

In [17]:
def gather_pred(model,loader,device):
    model.eval()
    total = 0
    all_preds = []
    all_labels = []
    all_gene = []
    all_pearson = []

    num_out=None
    loss_perlabel=None
    sq_error_perlabel=None
    abs_error_perlabel=None

    with torch.no_grad():
        for data, label, gene in loader:
            data = data.to(device)
            label = label.to(device).float().view(-1, 6)

            pred = model(data)
            if num_out is None:
                num_out = pred.shape[1]
                loss_perlabel = np.zeros(num_out,dtype=np.float32)
                sq_error_perlabel = np.zeros(num_out,dtype=np.float32)
                abs_error_perlabel = np.zeros(num_out,dtype=np.float32)
            loss = F.smooth_l1_loss(pred, label,reduction="none")

            diff = pred - label
            batch_size = label.size(0)
            total+=batch_size
            

            loss_perlabel += torch.sum(loss,dim=0).cpu().numpy()
            sq_error_perlabel += torch.sum(diff ** 2,dim=0).cpu().numpy()
            abs_error_perlabel += torch.sum(torch.abs(diff),dim=0).cpu().numpy()

            all_preds.append(pred.cpu().numpy())
            all_labels.append(label.cpu().numpy())
            all_gene.extend(gene)

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    avg_loss = loss_perlabel / max(total, 1)
    rmse = np.sqrt(sq_error_perlabel / max(total, 1))
    mae = abs_error_perlabel / max(total, 1)

    for i in range(all_preds.shape[1]):
        all_pearson.append(calc_pearson(all_labels[:, i], all_preds[:, i]))

    zhibiao_df = pd.DataFrame({
        "loss": avg_loss,
        "rmse": rmse,
        "mae": mae,
        "pearson": all_pearson
    })

    label_F_df=pd.DataFrame({"gene":all_gene})
    used_label=label_6
    for i,name in enumerate(used_label):
        label_F_df["label_"+name]=all_labels[:,i]
        label_F_df["pred_"+name]=all_preds[:, i]
    return zhibiao_df,label_F_df,all_gene

In [18]:
zhibiao_df,label_F_df,allgene=gather_pred(model,dl_test,device)

In [19]:
zhibiao_df.to_csv("zhibiao_rep1.csv",index=False)
label_F_df.to_csv("pred_result_rep1.csv",index=False)
